# Tutorial: Game NPC Persistent Memory Demo

This notebook shows how a game NPC can remember player details across sessions, retrieve relevant memories, and produce a compact payload for an engine adapter.


## Audience and Goal

Audience: game developers and technical artists evaluating Knoema as a social memory layer for NPCs.

Prerequisites: basic Python and the repository installed with `pip install -e .`.

By the end, you can store player-facing memories, retrieve them in a later scene, and convert the NPC action into a Godot-style payload.


## Outline

1. Import Knoema and create a temporary persistent memory database.
2. Define an NPC persona, player context, and relationship state.
3. Store session-one memories.
4. Retrieve those memories in session two.
5. Generate a dialogue action and an adapter payload.
6. Try one small exercise.


## 1. Setup

The demo uses a temporary SQLite/FAISS store and a deterministic local responder, so it runs without network access or API keys.


In [ ]:
import importlib.util
import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path
from tempfile import TemporaryDirectory

if importlib.util.find_spec('knoema') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.'])

from knoema import (
    Action,
    DecisionEngine,
    EmotionState,
    Environment,
    LocalClient,
    Memory,
    Persona,
    Personality,
    RelationshipGraph,
    SQLiteFaissMemoryStore,
)


## 2. NPC and Scene Context

The NPC is a clinic keeper in a small adventure game. The player returned after a previous scene.


In [ ]:
temp_dir = TemporaryDirectory()
db_path = Path(temp_dir.name) / 'npc_memory.sqlite3'

npc = Persona(
    agent_id='npc_ara',
    name='Ara',
    age=41,
    background='Clinic keeper in a coastal adventure game; remembers repeat visitors and small promises.',
    personality=Personality(
        openness=0.62,
        conscientiousness=0.81,
        extraversion=0.45,
        agreeableness=0.88,
        neuroticism=0.22,
    ),
    values=['care', 'discretion', 'practical help'],
    goals=['make the player feel remembered', 'offer useful local guidance'],
)

player_id = 'player_rin'
environment = Environment(
    start_time=datetime(2026, 5, 12, 18, 30),
    location_path=('Game World', 'North Harbor', 'Clinic'),
    conditions={'chapter': 2, 'weather': 'windy', 'shop_open': True},
)
environment.set_agent_location(npc.agent_id, ('Game World', 'North Harbor', 'Clinic Counter'))

relationships = RelationshipGraph()
relationships.add_agent(npc.agent_id)
relationships.add_agent(player_id)

npc.agent_id, str(db_path)


## 3. Store Session-One Memories

These memories represent facts the player revealed in an earlier visit.


In [ ]:
memory_store = SQLiteFaissMemoryStore(db_path)
session_one_time = datetime(2026, 5, 10, 16, 15)
session_one_memories = [
    Memory(
        id='npc-ara-memory-1',
        agent_id=npc.agent_id,
        timestamp=session_one_time,
        content='Rin said jasmine tea helps after long climbs.',
        memory_type='episodic',
        importance=0.82,
    ),
    Memory(
        id='npc-ara-memory-2',
        agent_id=npc.agent_id,
        timestamp=session_one_time,
        content='Rin lost a brass key near the tide pools and planned to ask around.',
        memory_type='episodic',
        importance=0.9,
    ),
    Memory(
        id='npc-ara-memory-3',
        agent_id=npc.agent_id,
        timestamp=session_one_time,
        content='Rin prefers direct hints instead of long tutorial dialogue.',
        memory_type='semantic',
        importance=0.75,
    ),
]

for memory in session_one_memories:
    memory_store.add(memory)

memory_store.close()
len(session_one_memories)


## 4. Retrieve Memories in a Later Session

Reopening the store simulates a new play session. Retrieval pulls the small set of memories relevant to the current scene.


In [ ]:
memory_store = SQLiteFaissMemoryStore(db_path)
retrieved_memories = memory_store.retrieve(
    'Rin returns to the clinic asking about the brass key and a warm drink',
    k=3,
)

[memory.content for memory in retrieved_memories]


## 5. Generate an NPC Action

The local responder reads the prompt context and returns one strict JSON action for the NPC.


In [ ]:
def npc_responder(messages):
    user_prompt = messages[-1]['content'].lower()
    if 'brass key' in user_prompt and 'jasmine tea' in user_prompt:
        content = 'Welcome back, Rin. I kept jasmine tea warm, and the brass key was turned in from the tide pools.'
    elif 'brass key' in user_prompt:
        content = 'Rin, someone brought in a brass key from the tide pools. Check the tray beside the register.'
    else:
        content = 'Good to see you again, Rin. Tell me what you need before the weather turns.'
    return json.dumps(
        {'action_type': 'speak', 'target': player_id, 'content': content},
        ensure_ascii=False,
    )

decision_engine = DecisionEngine(LocalClient(npc_responder))
emotion = EmotionState()
context = environment.get_context(npc.agent_id)
action = decision_engine.decide(
    persona=npc,
    memories=retrieved_memories,
    relationships=relationships.neighbors(npc.agent_id),
    environment=context,
    emotion=emotion.current,
    trigger=None,
)

action


## 6. Update Relationship State

A helpful remembered response can increase familiarity and trust before the next scene.


In [ ]:
relationship_action = Action(
    agent_id=npc.agent_id,
    timestamp=context.timestamp,
    action_type=action.action_type,
    target=player_id,
    content=action.content,
    location=action.location,
)
relationships.update_after_interaction(npc.agent_id, player_id, relationship_action, 'positive')
relationship = relationships.get_relationship(npc.agent_id, player_id)

{
    'type': relationship.relationship_type,
    'weight': round(relationship.weight, 3),
    'trust': round(relationship.trust, 3),
    'familiarity': round(relationship.familiarity, 3),
}


## 7. Adapter Payload

A game adapter can convert the action into a small payload for dialogue text, animation, and memory debugging.


In [ ]:
godot_payload = {
    'npc_id': action.agent_id,
    'target_id': action.target,
    'line': action.content,
    'location': action.location,
    'animation_hint': 'warm_greeting',
    'relationship': {
        'trust': round(relationship.trust, 3),
        'familiarity': round(relationship.familiarity, 3),
    },
    'memory_debug': [memory.id for memory in retrieved_memories],
}

godot_payload


In [ ]:
assert 'jasmine tea' in godot_payload['line']
assert 'brass key' in godot_payload['line']


In [ ]:
memory_store.close()
temp_dir.cleanup()


## Exercise

Add one new memory that the player dislikes long dialogue, then adjust `npc_responder` so the NPC gives a shorter line when that memory is retrieved.

Answer scaffold: add a fourth `Memory(...)`, rerun retrieval with `k=4`, and branch on a phrase like `direct hints` or `short dialogue` in `user_prompt`.


## Pitfall and Extension

Pitfall: storing every dialogue line forever can make retrieval noisy. Keep durable memories selective: player preferences, promises, conflicts, and world facts that matter later.

Extension: send `godot_payload` through the Godot adapter in `adapters/godot` and bind `animation_hint` to an NPC animation state.
